### Python Code for Simulation: A Simple $[\![5,1,3]\!]$ Example

In [ ]:
# Standard library imports
import pickle

# Third-party library imports
import numpy as np
import stim

# Local application/library imports
from lib import (
    compute_error_syndrome,
    compute_logical_phases,
    compute_symplectic_product,
    construct_resource_state,
)

# Constants
DATA_PATH = "../../data/qLDPC_codes/perfect5.pkl"

In [ ]:
# Load the saved parity check matrix and logical operators
with open(DATA_PATH, "rb") as file:
    parity_check_matrix, logical_x_matrix, logical_z_matrix, lookup_table = pickle.load(file)

# Ensure matrices are of type uint8 for efficient operations
parity_check_matrix = parity_check_matrix.astype(np.uint8, copy=False)
logical_x_matrix = logical_x_matrix.astype(np.uint8, copy=False)
logical_z_matrix = logical_z_matrix.astype(np.uint8, copy=False)

# Transform lookup table keys to tuples of np.uint8 and values to np.uint8 arrays
lookup_table = {
    tuple(np.array(key, dtype=np.uint8)): np.array(value, dtype=np.uint8)
    for key, value in lookup_table.items()
}

# Extract dimensions
num_logical_qubits = logical_z_matrix.shape[0]
num_checks = parity_check_matrix.shape[0]
num_physical_qubits = parity_check_matrix.shape[1] // 2
total_qubits = num_physical_qubits + num_logical_qubits

# Construct the tableau for the resource state
tableau = construct_resource_state(parity_check_matrix, logical_x_matrix, logical_z_matrix)

# Probability of depolarization
depol_error_rate = 0.1  # Probability of depolarization

# Initialize the circuit
# circuit = stim.Circuit()

circuit = (tableau + tableau).to_circuit()
circuit.append("DEPOLARIZE1", list(range(num_physical_qubits)), depol_error_rate)

# Create the circuit for measurement
for qubit_index in range(total_qubits):
    circuit.append("MXX", [qubit_index, qubit_index + total_qubits])
    circuit.append("MZZ", [qubit_index, qubit_index + total_qubits])

In [ ]:
# Probability of depolarization
depol_error_rate = 0.1  # Probability of depolarization

# Initialize the simulator
simulator = stim.TableauSimulator()

# Set the initial state from stabilizers
simulator.set_state_from_stabilizers(stabilizers)

# Apply depolarization to physical qubits
simulator.depolarize1(*range(num_physical_qubits), p=depol_error_rate)

# Execute the circuit
simulator.do_circuit(circuit)

# Extract and process measurement results
measurement_record = np.array(simulator.current_measurement_record(), dtype=np.uint8)
xx_input = measurement_record[::2][:num_physical_qubits]
zz_input = measurement_record[1::2][:num_physical_qubits]
xx_output = measurement_record[::2][num_physical_qubits:]
zz_output = measurement_record[1::2][num_physical_qubits:]

# Compute the error syndrome
error_syndrome = compute_error_syndrome(parity_check_matrix, xx_input, zz_input)

# Compute the phase of logical operators
logical_x_phase = compute_logical_phases(logical_x_matrix, xx_input, zz_input)
logical_z_phase = compute_logical_phases(logical_z_matrix, xx_input, zz_input)

# Verify the computed phase matches the measured phase
verification_passed = (
    np.all(logical_x_phase == xx_output) and np.all(logical_z_phase == zz_output)
)

# Print results
print("Verification Passed:", verification_passed)
print("Error Syndrome:", error_syndrome)
print("Measurement Record:", measurement_record)

# Retrieve decoded errors from the lookup table using the error syndrome
decoded_error = lookup_table[tuple(error_syndrome)]

# Compute phase corrections for logical X and Z operators
logical_x_phase_correction = compute_symplectic_product(logical_x_matrix, decoded_error)
logical_z_phase_correction = compute_symplectic_product(logical_z_matrix, decoded_error)

# Apply the corrections to the outputs
final_x_phase = (xx_output + logical_x_phase + logical_x_phase_correction) % 2
final_z_phase = (zz_output + logical_z_phase + logical_z_phase_correction) % 2

# Count the number of failures (non-zero entries in corrected phases)
failure_count = np.count_nonzero(np.bitwise_or(final_x_phase, final_z_phase))

# Print the error correction verification results
print("Failure Count:", failure_count)

### Monte Carol Simulation: A Simple $[\![5,1,3]\!]$ Example

In [ ]:
from lib.monte_carlo import MonteCarloBP

mc_sim = MonteCarloBP(
    parity_check_matrix=parity_check_matrix, 
    logical_x_matrix=logical_x_matrix, 
    logical_z_matrix=logical_z_matrix, 
    depol_error_rate=0.13, 
    decoder=lookup_table, target_simulations=10000)

mc_sim.run()

In [ ]:
from ldpc.mod2 import kernel

from matrix_helpers import quotient_basis_sp, canonicalize_ops

__all__ = [
    "compute_css_logical_operators",
]

def compute_css_logical_operators(x_parity_check_matrix, z_parity_check_matrix):
    """Calculates logical X and Z operators for a quantum code.

    Note:
        Logical operators derived from `x_parity_check_matrix` are logical Z operators.
        Logical operators derived from `z_parity_check_matrix` are logical X operators.

    Args:
        x_parity_check_matrix (np.ndarray): Parity-check matrix for X errors.
        z_parity_check_matrix (np.ndarray): Parity-check matrix for Z errors.

    Returns:
        tuple: A tuple (logical_x_matrix, logical_z_matrix) where:
            logical_x_matrix (np.ndarray): Logical X operator matrix.
            logical_z_matrix (np.ndarray): Logical Z operator matrix.
    """
    # Compute the kernel of the X parity-check matrix to derive logical Z operators
    x_kernel_matrix = kernel(x_parity_check_matrix)
    z_logical_operators = quotient_basis_sp(z_parity_check_matrix, x_kernel_matrix)

    # Compute the kernel of the Z parity-check matrix to derive logical X operators
    z_kernel_matrix = kernel(z_parity_check_matrix)
    x_logical_operators = quotient_basis_sp(x_parity_check_matrix, z_kernel_matrix)

    # Canonicalize the logical operators to ensure orthogonality
    canonical_x_matrix, canonical_z_matrix, _ = canonicalize_ops(x_logical_operators, z_logical_operators)

    return canonical_x_matrix, canonical_z_matrix

### qLDPC Entanglement Distillation Samplying by Indeterministic Measurements for Each Sample (Slow)

In [12]:
import pickle
import numpy as np
from css import compute_css_logical_operators


# Load the saved matrices
with open("/Users/aparnagupta/Downloads/notebooks/Measurement_based_distillation/ECC_gen/code_lib/hgp_code_lib/hgp_400_16_6.pkl", "rb") as f:
    code = pickle.load(f)
print(f"[[n={code['n']}, k={code['k']}, d<={code['d_est']}]]")
hgp_x, hgp_z = code["hgp_x"], code["hgp_z"]
#Ensure inputs are explicitly np.uint8
hgp_x = hgp_x.astype(np.uint8, copy=False)
hgp_z = hgp_z.astype(np.uint8, copy=False)

# Compute the logical X and Z operators that satisfy the canonical commutation relation
logical_x_matrix, logical_z_matrix = compute_css_logical_operators(hgp_x, hgp_z)


[[n=400, k=16, d<=6]]


In [15]:
from ldpc import BpOsdDecoder

from css_simulator import MonteCarloCSS

error_rate = 0.05

# Common decoder configuration
decoder_config = {
    "error_rate": error_rate * 2 / 3,
    "bp_method": "product_sum",
    "max_iter": 7,
    "schedule": "serial",
    "osd_method": "osd_cs",
    "osd_order": 2
}

# Initialize the Monte Carlo CSS simulator
mc_sim = MonteCarloCSS(
    parity_check_x_matrix=hgp_x,
    parity_check_z_matrix=hgp_z,
    logical_x_matrix=logical_x_matrix,
    logical_z_matrix=logical_z_matrix,
    depol_error_rate=error_rate,
    decoder_config=decoder_config,
    target_simulations=100
)

mc_sim.run()

Physical error rate: 5.00%; Logical error rate: 1.25 ± 1.11%: 100% 100/100 [00:11<00:00,  8.94it/s]


{'output_error_rate': 0.0125,
 'standard_deviation': 0.011110243021644487,
 'input_error_rate': 0.05,
 'simulation_count': 100,
 'failure_count': 20}

## Stim Circuit Sampling

We divided the simulation into two parts: the measurement sampling and post-precessing of the sampling.

The Stim circuit simulator incorporates noise effects directly into the measurement results. Consequently, samples are produced directly from these measurement results.

This approach allows measurement result samples to be efficiently generated with a single simulation of the circuit.

In [20]:
import numpy as np
import pickle
from tqdm import tqdm  # For progress tracking

from css_simulator import construct_css_resource_state

# Load the saved matrices
with open("/Users/aparnagupta/Downloads/notebooks/Measurement_based_distillation/ECC_gen/code_lib/hgp_code_lib/hgp_400_16_6.pkl", "rb") as f:
    code = pickle.load(f)
hgp_x, hgp_z = code["hgp_x"], code["hgp_z"]
# Ensure inputs are explicitly np.uint8
parity_check_x_matrix = hgp_x.astype(np.uint8, copy=False)
parity_check_z_matrix = hgp_z.astype(np.uint8, copy=False)

# Compute the logical X and Z operators that satisfy the canonical commutation relation
logical_x_matrix, logical_z_matrix = compute_css_logical_operators(parity_check_x_matrix, parity_check_z_matrix)

num_logical_qubits, num_physical_qubits = logical_x_matrix.shape
num_total_qubits = num_physical_qubits + num_logical_qubits

In [21]:
# Construct or Load the ideal circuit from disk

# Construct Circuit
# Compute the tableau representation for the resource state
tableau = construct_css_resource_state(
            parity_check_x_matrix.toarray(),
            parity_check_z_matrix.toarray(),
            logical_x_matrix.toarray(),
            logical_z_matrix.toarray()
        )

# Transform the tableau representation to a circuit for quick measurement sampling
circuit_init = (tableau + tableau).to_circuit()

In [ ]:
# # Save the Stim circuit to a file using pickle
# file_path = "_stim_circuit.pkl"

# with open(file_path, "wb") as file:
#     pickle.dump(circuit_init, file)


# # Load Circuit from the file
# file_path = "data/stim_circuits/circuit_1225_10.pkl"

# with open(file_path, "rb") as file:
#     circuit_init = pickle.load(file)

In [22]:
# Define depolarization error rates and their corresponding shot numbers
error_shots_dict = {
    0.009:  int(1e6),
    0.007:  int(2e6),
    0.006:  int(3e6),
    0.005:  int(4e6),
}

results_dict = {}

print("Starting sampling process...")

for depol_error_rate, num_shots in tqdm(error_shots_dict.items(), desc="Sampling Progress"):
    circuit = circuit_init.copy()
    circuit.append("DEPOLARIZE1", list(range(num_physical_qubits)), depol_error_rate)
    
    for qubit_index in range(num_total_qubits):
        target_qubit = qubit_index + num_total_qubits
        circuit.append("MXX", [qubit_index, target_qubit])
        circuit.append("MZZ", [qubit_index, target_qubit])
    
    sampler = circuit.compile_sampler()
    res = sampler.sample(shots=num_shots)  # Use the unique num_shots for this error rate
    measurements = np.array(res, dtype=np.uint8)
    
    results_dict[depol_error_rate] = {
        "num_shots": num_shots,
        "measurements": measurements
    }
    
    print(f"Completed sampling: depol_error_rate={depol_error_rate}, num_shots={num_shots}")

print("Sampling process complete.")

# Save all results in a structured dictionary format
file_path = "_samples.pkl"
with open(file_path, "wb") as file:
    pickle.dump(results_dict, file)

print("Sampling process complete. Data saved successfully.")

Starting sampling process...


Sampling Progress:  25%|██▌       | 1/4 [00:05<00:16,  5.60s/it]

Completed sampling: depol_error_rate=0.009, num_shots=1000000


Sampling Progress:  50%|█████     | 2/4 [00:16<00:17,  8.69s/it]

Completed sampling: depol_error_rate=0.007, num_shots=2000000


Sampling Progress:  75%|███████▌  | 3/4 [00:30<00:11, 11.19s/it]

Completed sampling: depol_error_rate=0.006, num_shots=3000000


Sampling Progress: 100%|██████████| 4/4 [00:50<00:00, 12.56s/it]

Completed sampling: depol_error_rate=0.005, num_shots=4000000
Sampling process complete.


Sampling process complete. Data saved successfully.


## Data Processing

This part processes the sampling data. Where we perform error correcting from the Bell measurements and compared the Bell measurements of the distilled states with their measurements.

### Defined Handy Functions

In [23]:
from ldpc import BpOsdDecoder

def initialize_bp_osd_decoders(parity_check_x_matrix, parity_check_z_matrix, depol_error_rate):
    """Initializes BP-OSD decoders for X and Z errors."""
    bp_osd_x = BpOsdDecoder(
        parity_check_x_matrix,
        error_rate=depol_error_rate * 2/3,
        bp_method='product_sum',
        max_iter=7,
        schedule='serial',
        osd_method='osd_cs',
        osd_order=2
    )

    bp_osd_z = BpOsdDecoder(
        parity_check_z_matrix,
        error_rate=depol_error_rate * 2/3,
        bp_method='product_sum',
        max_iter=7,
        schedule='serial',
        osd_method='osd_cs',
        osd_order=2
    )
    
    return bp_osd_x, bp_osd_z


def partition_measurements(measurement_data, num_physical_qubits):
    """Splits measurement records into input and output phases for X and Z."""
    xx_input = measurement_data[::2][:num_physical_qubits]
    zz_input = measurement_data[1::2][:num_physical_qubits]
    xx_output = measurement_data[::2][num_physical_qubits:]
    zz_output = measurement_data[1::2][num_physical_qubits:]
    return xx_input, zz_input, xx_output, zz_output

def apply_error_corrections(x_input, z_input, x_output, z_output, decoded_x_errors, decoded_z_errors):
    """Applies corrections to logical X and Z phases."""
    logical_x_phase = logical_x_matrix @ x_input % 2
    logical_z_phase = logical_z_matrix @ z_input % 2

    logical_x_correction = logical_x_matrix @ decoded_z_errors % 2
    logical_z_correction = logical_z_matrix @ decoded_x_errors % 2

    final_x_phase = (x_output + logical_x_phase + logical_x_correction) % 2
    final_z_phase = (z_output + logical_z_phase + logical_z_correction) % 2
    return final_x_phase, final_z_phase

### Processing

In [25]:
import numpy as np
import pickle
from tqdm import tqdm  # For progress tracking

# Load the error correction code matrices
with open("/Users/aparnagupta/Downloads/notebooks/Measurement_based_distillation/ECC_gen/code_lib/hgp_code_lib/hgp_400_16_6.pkl", "rb") as f:
    code = pickle.load(f)
H, hgp_x, hgp_z = code["H"], code["hgp_x"], code["hgp_z"]

# Load the sampling data
with open("/Users/aparnagupta/Downloads/notebooks/Measurement_based_distillation/distillation/_samples.pkl", "rb") as file:
    data = pickle.load(file)


# Ensure inputs are explicitly np.uint8
parity_check_x_matrix = hgp_x.astype(np.uint8, copy=False)
parity_check_z_matrix = hgp_z.astype(np.uint8, copy=False)

# Compute logical operators
from css import compute_css_logical_operators
logical_x_matrix, logical_z_matrix = compute_css_logical_operators(parity_check_x_matrix, parity_check_z_matrix)

num_logical_qubits, num_physical_qubits = logical_x_matrix.shape
num_total_qubits = num_physical_qubits + num_logical_qubits


# Dictionary to store results
results_dict = {}

# Process each depolarization error rate
for depol_error_rate, details in tqdm(data.items(), desc="Processing Data"):

    num_shots = details["num_shots"]
    measurements = details["measurements"]

    bp_osd_x, bp_osd_z = initialize_bp_osd_decoders(parity_check_x_matrix, parity_check_z_matrix, depol_error_rate)

    num_errors = 0

    # Iterate over each shot
    for shot in range(num_shots):
        xx_input, zz_input, xx_output, zz_output = partition_measurements(measurements[shot], num_physical_qubits)

        x_syndrome = parity_check_x_matrix @ xx_input % 2
        decoded_z_errors = bp_osd_x.decode(x_syndrome)

        z_syndrome = parity_check_z_matrix @ zz_input % 2
        decoded_x_errors = bp_osd_z.decode(z_syndrome)

        final_x_phase, final_z_phase = apply_error_corrections(
            xx_input, zz_input, xx_output, zz_output, decoded_x_errors, decoded_z_errors
        )

        num_errors += np.count_nonzero(np.bitwise_or(final_x_phase, final_z_phase))

    output_error_rate = num_errors / (num_logical_qubits * num_shots)
    standard_deviation = np.sqrt(
        output_error_rate * (1 - output_error_rate) / num_shots
    )

    print(f"Depolarization Error Rate: {100 * depol_error_rate:.2f}%")
    print(f"Logical Error Rate: {100 * output_error_rate:.3f} ± {100 * standard_deviation:.3f}%")

    # Store results
    results_dict[depol_error_rate] = {
        "num_shots": num_shots,
        "output_error_rate": output_error_rate,
        "standard_deviation": standard_deviation
    }

# Save the final results
output_file = "_processed_fidelity.pkl"
with open(output_file, "wb") as file:
    pickle.dump(results_dict, file)

print("Processing complete. Results saved.")

Processing Data:  25%|██▌       | 1/4 [03:50<11:30, 230.05s/it]

Depolarization Error Rate: 0.90%
Logical Error Rate: 0.007 ± 0.001%


Processing Data:  50%|█████     | 2/4 [10:46<11:19, 339.60s/it]

Depolarization Error Rate: 0.70%
Logical Error Rate: 0.004 ± 0.000%


Processing Data:  75%|███████▌  | 3/4 [20:35<07:33, 453.78s/it]

Depolarization Error Rate: 0.60%
Logical Error Rate: 0.002 ± 0.000%


Processing Data: 100%|██████████| 4/4 [32:52<00:00, 493.17s/it]

Depolarization Error Rate: 0.50%
Logical Error Rate: 0.001 ± 0.000%
Processing complete. Results saved.


## Data Combination

#### Functions

In [26]:
import numpy as np
import pickle
from typing import List, Dict

def combine_results(file_paths: List[str]) -> Dict[float, Dict[str, float]]:
    """Combines multiple error correction result files and returns the aggregated results.

    The function computes the weighted mean and standard deviation based on the total
    sampling pool across multiple simulation instances.

    Args:
        file_paths: List of file paths containing individual simulation results.

    Returns:
        Dict containing the aggregated results with the following structure:
        {
            depol_error_rate: {
                "num_shots": int,
                "output_error_rate": float,
                "standard_deviation": float
            },
            ...
        }
    """
    aggregated_results = {}

    # Load and aggregate results
    for file_path in file_paths:
        with open(file_path, "rb") as file:
            results = pickle.load(file)

        for error_rate, data in results.items():
            if error_rate not in aggregated_results:
                aggregated_results[error_rate] = {
                    "num_shots": [],
                    "output_error_rate": []
                }

            aggregated_results[error_rate]["num_shots"].append(data["num_shots"])
            aggregated_results[error_rate]["output_error_rate"].append(data["output_error_rate"])

    # Compute and store weighted statistics
    combined_results = {}
    for error_rate, data in aggregated_results.items():
        shots = np.array(data["num_shots"])
        errors = np.array(data["output_error_rate"])
        
        total_shots = np.sum(shots)
        weighted_mean = np.average(errors, weights=shots)
        weighted_std = np.sqrt(weighted_mean * (1 - weighted_mean) / total_shots)

        combined_results[error_rate] = {
            "num_shots": total_shots,
            "output_error_rate": weighted_mean,
            "standard_deviation": weighted_std
        }

    # return combined_results
    return dict(sorted(combined_results.items(), reverse=True))

#### Main

In [28]:
file_paths = [f"/Users/aparnagupta/Downloads/notebooks/Measurement_based_distillation/distillation/_processed_fidelity.pkl" for i in [1,2,3,4, 5, 6, 7, 8]]
merged_data = combine_results(file_paths)

# Example: Print combined results
for error_rate, data in merged_data.items():
    print(f"Input Error: {error_rate:.3f}, "
          f"Shots Num: {data['num_shots']:.1e}, "
          f"Output Error: {data['output_error_rate']:.1e}, "
          f"Std Dev: {100 * data['standard_deviation'] / data['output_error_rate']:.1f}%")
    
# Save the combined results as a new pickle file
output_file = "_merged_data.pkl"
with open(output_file, "wb") as file:
    pickle.dump(merged_data, file)

print(f"\n Merged data saved to \"{output_file}\"")

Input Error: 0.009, Shots Num: 8.0e+06, Output Error: 7.3e-05, Std Dev: 4.1%
Input Error: 0.007, Shots Num: 1.6e+07, Output Error: 3.9e-05, Std Dev: 4.0%
Input Error: 0.006, Shots Num: 2.4e+07, Output Error: 2.0e-05, Std Dev: 4.5%
Input Error: 0.005, Shots Num: 3.2e+07, Output Error: 1.3e-05, Std Dev: 4.9%

 Merged data saved to "_merged_data.pkl"


In [ ]:
using ForwardDiff

x = exp(-N)

function compute_params(d, x)
    sqrt_term = sqrt(1 + x^4 - 2d^2)

    a = (2x*d + (1 - x^2)*sqrt_term) / (1 + x^4)
    b = (x*sqrt_term - d*(1 - x^2)) / (1 + x^4)

    c = 0.5*(sqrt(1 + x^4 - 2d^2) + sqrt(1 - x^4))
    e = 0.5*(sqrt(1 + x^4 - 2d^2) - sqrt(1 - x^4))

    return a, b, c, e
end

function f(d, x)
    a, b, c, e = compute_params(d, x)
    return a*b - c*d
end

# Newton iteration
d = 0.1
for i in 1:50
    fp = ForwardDiff.derivative(d -> f(d, x), d)
    d -= f(d, x)/fp
end

a, b, c, e = compute_params(d, x)
Pe = 1 - (1/3)*(a^2 + 2c^2)

In [31]:
with open("//Users/aparnagupta/Downloads/notebooks/Measurement_based_distillation/distillation/_processed_fidelity.pkl", "rb") as file:
    data = pickle.load(file)


In [32]:
data

{0.009: {'num_shots': 1000000,
  'output_error_rate': 7.34375e-05,
  'standard_deviation': 8.569253580889863e-06},
 0.007: {'num_shots': 2000000,
  'output_error_rate': 3.928125e-05,
  'standard_deviation': 4.4316874316336e-06},
 0.006: {'num_shots': 3000000,
  'output_error_rate': 2.0333333333333334e-05,
  'standard_deviation': 2.603390090432658e-06},
 0.005: {'num_shots': 4000000,
  'output_error_rate': 1.29375e-05,
  'standard_deviation': 1.7984251875664546e-06}}